In [ ]:
# Repository-relative paths for the anonymized reproduction package.
import os
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analysis').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository tree.')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
MODULE_DIR = REPO_ROOT / 'analysis' / '05_be_meta_regression'
EXTERNAL_DATA_ROOT = Path(os.environ.get('HEATPA_DATA_ROOT', REPO_ROOT / 'external_data'))


In [ ]:
# -*- coding: utf-8 -*-
"""
Fig. 5a / 5c
Heterogeneity reduction by urban meta-predictor sets

Modified version:
- Figure canvas, dpi, font scaling and save settings are coordinated with the Fig. 5b
  exposure-lag surface code, so the two panels can be arranged side-by-side more easily.
- Mean-predictor red is changed to #a12d32.
- Gini-predictor blue is changed to #4c8eba.
- Error-bar / line transparency and point transparency are controlled by top-level parameters.
- Center points are enlarged, CI lines are thicker, the plotting area has a light-grey base,
  the central reference dashed line is visually stronger, only the region left of that
  dashed line is shaded light grey, and value labels are placed above the lines
  to avoid covering the symbols.
- The main data frame now uses the same GridSpec width/height and right-panel reserve as Fig. 5b.

Input:
    fig5c_plot_data.csv

Expected columns:
    activity_label
    predictor_mode
    heterogeneity_explained_pct
    loo_ci_low
    loo_ci_high

Run:
    python fig5c_make_nature_figure_matched_size_data_path_fixed_argv.py

    也可以传入 fig5c 工程文件夹或 data 文件夹，例如：
    python fig5c_make_nature_figure_matched_size_data_path_fixed_argv.py "analysis_code/05_be_meta_regression"

Default input:
    analysis_code/05_be_meta_regression/data/figure5_c/fig5c_plot_data.csv

Default output:
    analysis_code/05_be_meta_regression/output/fig5c_heterogeneity_reduction_nature_fig5b_frame_matched.png
    analysis_code/05_be_meta_regression/output/fig5c_heterogeneity_reduction_nature_fig5b_frame_matched.svg
    analysis_code/05_be_meta_regression/output/fig5c_heterogeneity_reduction_nature_fig5b_frame_matched.pdf
"""

from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba
import numpy as np
import pandas as pd


# =========================================================
# 1. 路径与输出
# =========================================================

# 默认基础数据文件夹：采用截图1中的 Source Data 文件夹。
# 基础数据默认读取：
#   analysis_code/05_be_meta_regression/data/figure5_c/fig5c_plot_data.csv
# 输出默认保存到截图2中的 Figure 文件夹：
#   analysis_code/05_be_meta_regression/output
#
# 注意：在 Jupyter / Spyder / VS Code notebook 中，sys.argv 可能自动包含
# --ip=127.0.0.1 等内核参数。旧代码会误把 --ip=127.0.0.1 当作 WORK，
# 导致报错：--ip=127.0.0.1\data\fig5c_plot_data.csv。
# 因此这里仅在命令行参数看起来像真实路径时才采用该参数。
DEFAULT_DATA_DIR = Path(
    str(MODULE_DIR / "data" / "figure5_c")
)
DEFAULT_OUT_DIR = Path(
    str(MODULE_DIR / "output")
)


def _is_real_user_path_arg(arg: str) -> bool:
    """判断 sys.argv 中的参数是否像用户手动传入的路径。"""
    if not arg:
        return False
    s = str(arg).strip().strip('"').strip("'")

    # Jupyter / IPython / Spyder 常见内核参数，例如 --ip=127.0.0.1、-f xxx.json。
    if s.startswith("-"):
        return False
    if s.endswith(".json"):
        return False

    # Windows 盘符路径、UNC 路径、Linux/Mac 绝对路径、或含目录分隔符的相对路径。
    if (len(s) >= 3 and s[1] == ":" and s[2] in ["\\", "/"]):
        return True
    if s.startswith("\\") or s.startswith("/"):
        return True
    if "\\" in s or "/" in s:
        return True

    return False


def _get_user_path_from_argv() -> Path | None:
    """从命令行参数中提取用户真正传入的路径；忽略 notebook 内核参数。"""
    for arg in sys.argv[1:]:
        if _is_real_user_path_arg(arg):
            return Path(str(arg).strip().strip('"').strip("'"))
    return None


USER_PATH = _get_user_path_from_argv()

if USER_PATH is None:
    DATA_DIR = DEFAULT_DATA_DIR
    OUT = DEFAULT_OUT_DIR
else:
    # 兼容两种传参：
    # 1) 传入 fig5c 工程文件夹：...\fig5c
    # 2) 直接传入 data 文件夹：...\fig5c\data
    if USER_PATH.name.lower() == "data":
        DATA_DIR = USER_PATH
        OUT = USER_PATH.parent
    else:
        DATA_DIR = USER_PATH / "data"
        OUT = USER_PATH

DATA = DATA_DIR / "fig5c_plot_data.csv"

# 如需把图件直接输出到 data 文件夹，可改为：OUT = DATA_DIR
OUT.mkdir(parents=True, exist_ok=True)

OUT_STEM = "fig5c_heterogeneity_reduction_nature_fig5b_frame_matched"


# =========================================================
# 2. 图像尺寸：与 Fig. 5b 代码相称
# =========================================================

# 与第二个代码保持一致：FIGSIZE = (9.4, 5.9)，DPI = 600。
# 如果后续要在 Word / AI / PPT 中并排排版，可优先保持两个代码的 FIGSIZE 和 DPI 一致。
FIGSIZE = (9.4, 5.9)
DPI = 600

# 是否使用与第二个代码一致的 tight bbox 保存方式。
# True：导出边缘更紧，适合论文排版；False：严格保留 FIGSIZE 画布。
SAVE_WITH_TIGHT_BBOX = True
SAVE_PAD_INCHES = 0.04


# =========================================================
# 3. 字体与 Nature 风格参数
# =========================================================

FONT_FAMILY = "Arial"

# 与第二个代码一致的自适应字体缩放逻辑。
AUTO_FONT_SCALE = True
REFERENCE_FIG_WIDTH = 8.8
MIN_FONT_SCALE = 1.06
MAX_FONT_SCALE = 1.28

TITLE_SIZE_BASE = 14.0
SUBTITLE_SIZE_BASE = 11.0
AXIS_LABEL_SIZE_BASE = 11.5
TICK_SIZE_BASE = 10.0
PANEL_LABEL_SIZE_BASE = 20.0
LEGEND_FONT_SIZE_BASE = 8.2
VALUE_LABEL_SIZE_BASE = 8.0
FOOTNOTE_SIZE_BASE = 7.0

# =========================================================
# 3.1 主数据框布局：严格对齐第二个 Fig. 5b 代码
# =========================================================

# True：使用第二个代码相同的 GridSpec 布局，即主绘图区 + 右侧占位面板。
# 这样第一个代码的“数据框 / 主 axes”实际长宽会与第二个代码一致。
# False：退回单 axes 布局，主图会更宽，不再与第二个代码的数据框完全一致。
MATCH_FIG5B_MAIN_FRAME = True

# 与第二个代码一致：主图框比例 height / width。
SET_MAIN_AX_BOX_ASPECT = True
MAIN_AX_BOX_ASPECT = 0.72

# 与第二个代码一致：右侧整体面板宽度和主图-右侧面板间距。
# 即使本图没有 colorbar，也保留这个右侧面板，用于放 legend 或作为占位，
# 从而使左侧主数据框宽度与 Fig. 5b 完全一致。
RIGHT_PANEL_WIDTH_RATIO = 0.22
RIGHT_PANEL_WSPACE = 0.075

# 与第二个代码一致的整体边距。
# 注意：如果你手动增大 FIG_LEFT，左侧 y 轴标签更宽松，但主数据框位置将不再与第二个代码完全一致。
FIG_LEFT = 0.085
FIG_RIGHT = 0.965
FIG_BOTTOM = 0.135
FIG_TOP = 0.865

# 右侧 legend 位置，坐标为 right_ax 内部的相对坐标：[x0, y0, width, height]。
# 放到右侧面板后，主图框不会被 legend 挤压或改变宽度。
USE_RIGHT_PANEL_FOR_LEGEND = True
RIGHT_LEGEND_BOX = [0.00, 0.58, 0.98, 0.28]

# legend 样式：与第二个代码一致。
LEGEND_HANDLE_LENGTH = 2.1
LEGEND_LABEL_SPACING = 0.75
LEGEND_HANDLE_TEXT_PAD = 0.65

SHOW_PANEL_LABEL = True
PANEL_LABEL = "c"   # 与第二个代码一致；若作为 Fig. 5a 使用，可改为 "a"

TITLE_TEXT = "Heterogeneity reduction by urban meta-predictor sets"
SUBTITLE_TEXT = "National | CEHWI | Composite | 63 cities"
X_LABEL = "Between-city heterogeneity reduction (%)"
FOOTNOTE_TEXT = (
    r"$R^2_{I^2}=1-I^2_{\mathrm{model}}/I^2_{\mathrm{null}}$; "
    r"intervals are leave-one-city-out empirical 95% ranges."
)


def _font_scale() -> float:
    if not AUTO_FONT_SCALE:
        return 1.0
    scale = FIGSIZE[0] / REFERENCE_FIG_WIDTH
    return max(MIN_FONT_SCALE, min(MAX_FONT_SCALE, scale))


_FONT_SCALE = _font_scale()

TITLE_SIZE = TITLE_SIZE_BASE * _FONT_SCALE
SUBTITLE_SIZE = SUBTITLE_SIZE_BASE * _FONT_SCALE
AXIS_LABEL_SIZE = AXIS_LABEL_SIZE_BASE * _FONT_SCALE
TICK_SIZE = TICK_SIZE_BASE * _FONT_SCALE
PANEL_LABEL_SIZE = PANEL_LABEL_SIZE_BASE * _FONT_SCALE
LEGEND_FONT_SIZE = LEGEND_FONT_SIZE_BASE * _FONT_SCALE
VALUE_LABEL_SIZE = VALUE_LABEL_SIZE_BASE * _FONT_SCALE
FOOTNOTE_SIZE = FOOTNOTE_SIZE_BASE * _FONT_SCALE


# =========================================================
# 4. 颜色、线条、点透明度
# =========================================================

PREDICTOR_COLORS = {
    "Mean predictors": "#a12d32",
    "Gini predictors": "#4c8eba",
}

PREDICTOR_MARKERS = {
    "Mean predictors": "o",
    "Gini predictors": "s",
}

# 两组点在同一活动行中的上下偏移。
PREDICTOR_Y_OFFSETS = {
    "Mean predictors": 0.16,
    "Gini predictors": -0.16,
}

# 线条透明度：控制水平置信区间线、端帽、legend 线段。
LINE_ALPHA = 0.8

# 点透明度：控制红/蓝中心点。
POINT_ALPHA =1.0

# 数值标注透明度。一般不需要很透明，避免读数困难。
VALUE_LABEL_ALPHA = 0.96

# 增大点与线的视觉权重。POINT_SIZE 以 point 为单位，scatter 中会自动平方。
ERRORBAR_LINE_WIDTH = 2.20
ERRORBAR_CAP_SIZE = 4.2
ERRORBAR_CAP_THICKNESS = 1.80
POINT_SIZE = 7.2
POINT_EDGE_WIDTH = 0.65

# 画布与绘图区保持白色；灰色只加在参考虚线左侧区域。
FIGURE_FACE_COLOR = "white"
AXES_FACE_COLOR = "white"

# 参考虚线位置与左侧灰底。与第二个代码一致，默认绘制 0% 参考线。
REFERENCE_LINE_X = 0.0
LEFT_SHADE_COLOR = "#F0F0F0"
LEFT_SHADE_ALPHA = 0.6

# 中间参考虚线：加深颜色并略加粗。
ZERO_LINE_COLOR = "#7F7F7F"
ZERO_LINE_WIDTH = 1.25
ZERO_LINE_ALPHA = 1.00
ZERO_LINE_STYLE = (0, (2.5, 2.5))

GRID_X_COLOR = "#E8E3DF"
GRID_Y_COLOR = "#F1EEEA"
GRID_X_WIDTH = 0.95
GRID_Y_WIDTH = 0.95

SPINE_COLOR = "#555555"
SPINE_WIDTH = 1.0

# 数值标注放在线条上方，避免遮挡，不使用白底框。
VALUE_LABEL_VERTICAL_OFFSET = 0.08
TEXT_BBOX_ALPHA = 0.0

# 是否使用第二个代码中的“侧向标签 + 白底框”样式。
# False：保留当前版本“标签位于线条上方”的样式，避免遮挡中心点；
# True：恢复第二个代码的标签位置和白底框。
USE_SECOND_CODE_SIDE_VALUE_LABEL = False
VALUE_LABEL_SIDE_DX = 0.18
VALUE_LABEL_SIDE_BBOX_ALPHA = 0.72


# =========================================================
# 5. 数据与坐标设置
# =========================================================

ACTIVITY_ORDER = ["All activity", "Cycling", "Running", "Walking"]
PREDICTOR_ORDER = ["Mean predictors", "Gini predictors"]

# x 轴留白比例。若标注贴边，可适当增大。
X_PAD_FRACTION = 0.22
X_PAD_MIN = 0.8
X_MIN_SPAN = 6.0


# =========================================================
# 6. 样式函数
# =========================================================


def set_nature_style() -> None:
    plt.rcParams.update(
        {
            "font.family": FONT_FAMILY,
            "font.size": TICK_SIZE,
            "axes.titlesize": TITLE_SIZE,
            "axes.labelsize": AXIS_LABEL_SIZE,
            "xtick.labelsize": TICK_SIZE,
            "ytick.labelsize": TICK_SIZE,
            "legend.fontsize": LEGEND_FONT_SIZE,
            "svg.fonttype": "none",
            "pdf.fonttype": 42,
            "axes.unicode_minus": False,
            "axes.linewidth": SPINE_WIDTH,
            "xtick.major.width": 0.9,
            "ytick.major.width": 0.9,
            "xtick.major.size": 4.5,
            "ytick.major.size": 0,
        }
    )


def nice_limits(vals: np.ndarray) -> tuple[float, float]:
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return -5.0, 5.0

    lo = min(float(vals.min()), 0.0)
    hi = max(float(vals.max()), 0.0)
    pad = max(X_PAD_MIN, (hi - lo) * X_PAD_FRACTION)

    lo = np.floor((lo - pad) / 1.0) * 1.0
    hi = np.ceil((hi + pad) / 1.0) * 1.0

    if hi - lo < X_MIN_SPAN:
        mid = (hi + lo) / 2.0
        lo, hi = mid - X_MIN_SPAN / 2.0, mid + X_MIN_SPAN / 2.0

    return lo, hi


def _save_figure(fig: plt.Figure, out_dir: Path, stem: str) -> None:
    for ext in ("svg", "pdf", "png"):
        path = out_dir / f"{stem}.{ext}"
        save_kwargs = {}

        if ext == "png":
            save_kwargs["dpi"] = DPI

        if SAVE_WITH_TIGHT_BBOX:
            save_kwargs["bbox_inches"] = "tight"
            save_kwargs["pad_inches"] = SAVE_PAD_INCHES

        fig.savefig(path, **save_kwargs)


# =========================================================
# 7. 主程序
# =========================================================


def main() -> None:
    set_nature_style()

    if not DATA.exists():
        raise FileNotFoundError(
            f"未找到输入数据：{DATA}\n"
            "Verify that the module data directory contains fig5c_plot_data.csv.\n"
            "当前默认 DATA_DIR 为：" + str(DATA_DIR)
        )

    df = pd.read_csv(DATA)

    required_cols = [
        "activity_label",
        "predictor_mode",
        "heterogeneity_explained_pct",
        "loo_ci_low",
        "loo_ci_high",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"输入数据缺少必要列：{missing}")

    df = df.copy()

    # 保留第二个代码的顺序：All activity、Cycling、Running、Walking。
    # 若数据中存在额外活动或拼写错误，这里直接提示，避免后续 KeyError 难以定位。
    unknown_activity = sorted(set(df["activity_label"].dropna().astype(str)) - set(ACTIVITY_ORDER))
    unknown_predictor = sorted(set(df["predictor_mode"].dropna().astype(str)) - set(PREDICTOR_ORDER))
    if unknown_activity:
        raise ValueError(f"activity_label 中存在未配置的类别：{unknown_activity}")
    if unknown_predictor:
        raise ValueError(f"predictor_mode 中存在未配置的类别：{unknown_predictor}")

    df["activity_label"] = pd.Categorical(
        df["activity_label"], categories=ACTIVITY_ORDER, ordered=True
    )
    df["predictor_mode"] = pd.Categorical(
        df["predictor_mode"], categories=PREDICTOR_ORDER, ordered=True
    )
    df = df.sort_values(["activity_label", "predictor_mode"])

    y_base = {name: len(ACTIVITY_ORDER) - 1 - i for i, name in enumerate(ACTIVITY_ORDER)}

    # 使用与第二个代码相同的 1×2 GridSpec 布局。
    # 左侧 ax 是真正的数据框；右侧 right_ax 只用于 legend / 占位，
    # 这样左侧数据框的长宽与 Fig. 5b 主图框一致。
    fig = plt.figure(figsize=FIGSIZE)
    fig.patch.set_facecolor(FIGURE_FACE_COLOR)

    if MATCH_FIG5B_MAIN_FRAME:
        gs = fig.add_gridspec(
            nrows=1,
            ncols=2,
            width_ratios=[1.0, RIGHT_PANEL_WIDTH_RATIO],
            wspace=RIGHT_PANEL_WSPACE,
        )
        ax = fig.add_subplot(gs[0, 0])
        right_ax = fig.add_subplot(gs[0, 1])
        right_ax.axis("off")

        leg_ax = right_ax.inset_axes(RIGHT_LEGEND_BOX)
        leg_ax.axis("off")
    else:
        ax = fig.add_subplot(111)
        right_ax = None
        leg_ax = None

    ax.set_facecolor(AXES_FACE_COLOR)

    if SET_MAIN_AX_BOX_ASPECT:
        ax.set_box_aspect(MAIN_AX_BOX_ASPECT)

    all_x: list[float] = []

    for mode in PREDICTOR_ORDER:
        sub = df[df["predictor_mode"] == mode].copy()
        if sub.empty:
            continue

        x = sub["heterogeneity_explained_pct"].to_numpy(dtype=float)
        lo = sub["loo_ci_low"].to_numpy(dtype=float)
        hi = sub["loo_ci_high"].to_numpy(dtype=float)
        y = (
            np.array([y_base[str(v)] for v in sub["activity_label"]], dtype=float)
            + PREDICTOR_Y_OFFSETS[mode]
        )

        all_x.extend(x.tolist())
        all_x.extend(lo.tolist())
        all_x.extend(hi.tolist())

        xerr = np.vstack([x - lo, hi - x])
        color = PREDICTOR_COLORS[mode]

        # 先绘制水平 CI / 线条，透明度由 LINE_ALPHA 控制。
        ax.errorbar(
            x,
            y,
            xerr=xerr,
            fmt="none",
            elinewidth=ERRORBAR_LINE_WIDTH,
            capsize=ERRORBAR_CAP_SIZE,
            capthick=ERRORBAR_CAP_THICKNESS,
            ecolor=to_rgba(color, LINE_ALPHA),
            zorder=3,
        )

        # 再单独绘制中心点，透明度由 POINT_ALPHA 控制。
        ax.scatter(
            x,
            y,
            s=POINT_SIZE ** 2,
            marker=PREDICTOR_MARKERS[mode],
            facecolors=to_rgba(color, POINT_ALPHA),
            edgecolors=to_rgba("white", POINT_ALPHA),
            linewidths=POINT_EDGE_WIDTH,
            zorder=4,
        )

        # 数值标注：默认放在线条上方，避免遮挡点和误差线；
        # 也可通过 USE_SECOND_CODE_SIDE_VALUE_LABEL 切换为第二个代码的侧向白底标签。
        for xi, yi in zip(x, y):
            if USE_SECOND_CODE_SIDE_VALUE_LABEL:
                ha = "left" if xi >= REFERENCE_LINE_X else "right"
                dx = VALUE_LABEL_SIDE_DX if xi >= REFERENCE_LINE_X else -VALUE_LABEL_SIDE_DX
                ax.text(
                    xi + dx,
                    yi,
                    f"{xi:+.1f}%",
                    va="center",
                    ha=ha,
                    color=to_rgba(color, VALUE_LABEL_ALPHA),
                    fontsize=VALUE_LABEL_SIZE,
                    bbox=dict(
                        fc="white",
                        ec="none",
                        alpha=VALUE_LABEL_SIDE_BBOX_ALPHA,
                        pad=0.25,
                    ),
                    zorder=5,
                )
            else:
                ax.text(
                    xi,
                    yi + VALUE_LABEL_VERTICAL_OFFSET,
                    f"{xi:+.1f}%",
                    va="bottom",
                    ha="center",
                    color=to_rgba(color, VALUE_LABEL_ALPHA),
                    fontsize=VALUE_LABEL_SIZE,
                    zorder=5,
                )

    xmin, xmax = nice_limits(np.asarray(all_x, dtype=float))
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(-0.62, len(ACTIVITY_ORDER) - 0.38)

    ax.set_yticks([y_base[v] for v in ACTIVITY_ORDER])
    ax.set_yticklabels(ACTIVITY_ORDER)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_SIZE)

    # 仅在参考虚线左侧添加浅灰底，而不是给整张图加灰底。
    shade_left = xmin
    shade_right = min(REFERENCE_LINE_X, xmax)
    if shade_right > shade_left:
        ax.axvspan(
            shade_left,
            shade_right,
            facecolor=LEFT_SHADE_COLOR,
            alpha=LEFT_SHADE_ALPHA,
            zorder=0,
            ec="none",
        )

    ax.axvline(
        REFERENCE_LINE_X,
        color=to_rgba(ZERO_LINE_COLOR, ZERO_LINE_ALPHA),
        lw=ZERO_LINE_WIDTH,
        ls=ZERO_LINE_STYLE,
        zorder=2,
    )

    ax.grid(axis="x", color=GRID_X_COLOR, lw=GRID_X_WIDTH)
    ax.grid(axis="y", color=GRID_Y_COLOR, lw=GRID_Y_WIDTH)

    ax.tick_params(axis="both", labelsize=TICK_SIZE, length=4.5, width=0.9)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(SPINE_COLOR)
    ax.spines["bottom"].set_color(SPINE_COLOR)
    ax.spines["left"].set_linewidth(SPINE_WIDTH)
    ax.spines["bottom"].set_linewidth(SPINE_WIDTH)

    # 自定义 legend，使线条和点分别继承透明度设置。
    legend_handles = []
    for mode in PREDICTOR_ORDER:
        if mode not in set(df["predictor_mode"].dropna().astype(str)):
            continue
        color = PREDICTOR_COLORS[mode]
        legend_handles.append(
            Line2D(
                [0],
                [0],
                color=to_rgba(color, LINE_ALPHA),
                lw=ERRORBAR_LINE_WIDTH,
                marker=PREDICTOR_MARKERS[mode],
                markersize=POINT_SIZE,
                markerfacecolor=to_rgba(color, POINT_ALPHA),
                markeredgecolor=to_rgba("white", POINT_ALPHA),
                markeredgewidth=POINT_EDGE_WIDTH,
                label=mode,
            )
        )

    if USE_RIGHT_PANEL_FOR_LEGEND and leg_ax is not None:
        # 与第二个代码一致：legend 放入右侧面板，避免改变主数据框尺寸。
        leg_ax.legend(
            handles=legend_handles,
            frameon=False,
            loc="upper left",
            bbox_to_anchor=(0.0, 1.0),
            borderaxespad=0.0,
            handlelength=LEGEND_HANDLE_LENGTH,
            handletextpad=LEGEND_HANDLE_TEXT_PAD,
            labelspacing=LEGEND_LABEL_SPACING,
            fontsize=LEGEND_FONT_SIZE,
        )
    else:
        # 备用：保留原来的主图上方横向 legend。
        ax.legend(
            handles=legend_handles,
            frameon=False,
            loc="lower right",
            bbox_to_anchor=(1.0, 1.055),
            ncol=2,
            handlelength=1.8,
            columnspacing=1.1,
            borderaxespad=0.0,
            fontsize=LEGEND_FONT_SIZE,
        )

    if SHOW_PANEL_LABEL:
        fig.text(
            0.045,
            0.935,
            PANEL_LABEL,
            fontsize=PANEL_LABEL_SIZE,
            fontweight="bold",
            ha="left",
            va="top",
        )

    fig.text(
        0.17,
        0.925,
        TITLE_TEXT,
        fontsize=TITLE_SIZE,
        fontweight="bold",
        ha="left",
        va="top",
    )
    fig.text(
        0.17,
        0.872,
        SUBTITLE_TEXT,
        fontsize=SUBTITLE_SIZE,
        color="#666666",
        ha="left",
        va="top",
    )
    fig.text(
        0.965,
        0.055,
        FOOTNOTE_TEXT,
        ha="right",
        va="bottom",
        fontsize=FOOTNOTE_SIZE,
        color="#555555",
    )

    fig.subplots_adjust(
        left=FIG_LEFT,
        right=FIG_RIGHT,
        bottom=FIG_BOTTOM,
        top=FIG_TOP,
    )

    _save_figure(fig, OUT, OUT_STEM)
    plt.close(fig)

    print("Input data:")
    print(DATA)
    print("Output folder:")
    print(OUT)
    print("Figure saved:")
    for ext in ("png", "svg", "pdf"):
        print(OUT / f"{OUT_STEM}.{ext}")

    print("Figure size:")
    print(FIGSIZE)
    print("DPI:")
    print(DPI)
    print("Transparency:")
    print(f"  line alpha = {LINE_ALPHA}")
    print(f"  point alpha = {POINT_ALPHA}")
    print("Reference region:")
    print(f"  reference line x = {REFERENCE_LINE_X}")
    print(f"  left shade color = {LEFT_SHADE_COLOR}")
    print("Matched main-frame layout:")
    print(f"  match Fig. 5b frame = {MATCH_FIG5B_MAIN_FRAME}")
    print(f"  main axis box aspect = {MAIN_AX_BOX_ASPECT}")
    print(f"  right panel width ratio = {RIGHT_PANEL_WIDTH_RATIO}")
    print(f"  right panel wspace = {RIGHT_PANEL_WSPACE}")
    print(f"  figure margins = left {FIG_LEFT}, right {FIG_RIGHT}, bottom {FIG_BOTTOM}, top {FIG_TOP}")
    print("Colors:")
    print(f"  Mean predictors = {PREDICTOR_COLORS['Mean predictors']}")
    print(f"  Gini predictors = {PREDICTOR_COLORS['Gini predictors']}")


if __name__ == "__main__":
    main()
